# 8j — Preliminary composable forecast (four ways)

Implements `inst/1a_preliminary_framework_plan.md` + the fixes/extensions in `inst/1c`,
`inst/1d`, `inst/1e`: a joint renewal / next-generation-matrix model driven by **age-pair
contact-degree distributions**, scored **four ways** — the 2×2 grid of {unweighted
**NegBin**, weighted **Hurdle-Weibull**} degree models × {**Mean**, **Neighbourhood**} NGM.

The contact **mean** is estimated with **structural reciprocity** (`log μ_{i→j} = r + log Nⱼ`)
and **spatial-GP smoothing** across the age-pair grid (separable RBF, shared length-scale;
inst/1e). Forecasts use the **contact-updated iterate** over **4 origins** × 4 horizons;
**WIS** is computed on a **log scale** and aggregated **by horizon** via R `scoringutils`.

Fitting uses **Pathfinder.jl** (parsimonious fit / init) and optionally **Turing NUTS** (`USE_NUTS`).
See the specs for modelling details and the remaining lean simplifications (constant
within-window contacts; reduced transmission block).

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

USE_NUTS = false   # true ⇒ formal Turing NUTS fit (slow); false ⇒ Pathfinder parsimonious fit

## §1 Window, infection/antibody data, and age-pair degree data

In [ ]:
cfg  = FrameworkConfig()
grid = cis_age_grid()

# Read the CoMix contact data ONCE and reuse it across every window (avoids re-reading/
# re-joining the full Arrow per origin×horizon). Then roll the forecast origin over the
# whole period the current datasets support ("available period"): each origin needs a
# 12-week fit/lag window back to the first inc2prev week, and contact data out to
# origin+3 for the contact-updated iterate. `available_forecast_origins` derives the range.
raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS))
let w = wins[1], wd0 = load_window_data(wins[1]; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

In [ ]:
# Fit config: the four combos and the parallel-fit concurrency (CPU- and memory-balanced).
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

## §2 Roll over the available period — fit four ways, forecast 1–4 weeks ahead

For **each weekly origin** across the available period the four combos are fit and forecast.
The contact **mean** is a **reciprocity-structural, GP-smoothed** field (one symmetric
log-rate per unordered age pair, separable-RBF smoothing over age midpoints 70+→74.5,
`log μ_{i→j}=r+log Nⱼ` ⟹ exact reciprocity). Forecasting is the **contact-updated iterate**
(inst/1d): per origin t₀ and horizon `h`, the degree window ends at `t₀+h−1` (infections/
antibody frozen at t₀), the NGM is refreshed and one renewal step taken.

The loop is **memory-bounded**: per origin it builds only that origin's 4 degree windows
(reusing the single raw read), **parallel-pre-fits** its 16 chains (`prefit_chains!`, bounded
to `MAX_FIT_CONCURRENCY` simultaneous fits), assembles the forecasts, then discards the degree
data. Chains are cached per (origin, horizon) under `../dt_intermediate/8j_chn_*.jld2`, so the
run is **resumable** — a re-run reloads finished chains and only fits what's missing.

In [ ]:
qtabs     = DataFrame[]
fc_store  = Dict{Tuple{Date,String},Array{Float64,3}}()
crps_rows = NamedTuple[]
skipped   = Tuple{Date,String}[]
t0 = time()
for (oi, win_o) in enumerate(wins)
    wd_o    = load_window_data(win_o; grid = grid)
    truth_o = load_forecast_truth(win_o; grid = grid)
    # this origin's 4 contact/degree windows (reuse the single raw read); discarded after.
    apd_o = [prepare_degree_data(
                 WeeklyWindow(win_o.origin + Day(7 * (h - 1));
                              n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
                 cfg; grid = grid, setting = :all,
                 df_part_raw = raw.df_part, craw_raw = raw.craw)
             for h in cfg.horizons]
    # parallel pre-fit this origin's 16 chains (cached ones skipped)
    prefit_chains!(combos, [win_o], [wd_o], cfg, [apd_o];
                   grid = grid, setting = :all, use_nuts = USE_NUTS,
                   save_dir = "../dt_intermediate", max_concurrent = MAX_FIT_CONCURRENCY)
    for (dm, nb) in combos
        lbl = string(degree_label(dm), "|", ngm_label(nb))
        try   # keep the multi-origin run alive if a single origin×combo fit is pathological
            fc = iterated_forecast(dm, nb, wd_o, cfg, win_o;
                                   grid = grid, setting = :all, use_nuts = USE_NUTS,
                                   save_dir = "../dt_intermediate", apd_by_h = apd_o)
            fc_store[(win_o.origin, lbl)] = fc
            push!(qtabs, to_quantile_long(fc, truth_o, lbl, win_o, cfg, grid.LAB))
            push!(crps_rows, (origin = win_o.origin, model = lbl, mean_crps = mean_crps(fc, truth_o)))
        catch err
            push!(skipped, (win_o.origin, lbl))
            @warn "skipped origin×combo" origin=win_o.origin model=lbl exception=err
        end
    end
    if oi % 5 == 0 || oi == length(wins)
        println("  origin $oi/$(length(wins)) (", win_o.origin, ")  elapsed ",
                round(Int, time() - t0), "s")
    end
end
qall = vcat(qtabs...)              # all origins × combos, tagged by forecast_date=origin
println("quantile rows: ", size(qall), "   (", length(wins), " origins × ", length(combos),
        " combos; skipped ", length(skipped), ")")
size(qall)

## §3 Score four ways — WIS via `scoringutils`

In [ ]:
scores = score_wis(qall)   # scores both natural & log scale; aggregated by horizon (inst/1e)

# Headline: LOG-SCALE WIS aggregated BY HORIZON across all origins.
by_mh_log = sort(@subset(scores.by_model_h, :scale .== "log"), [:model, :horizon])
by_m_log  = sort(@subset(scores.by_model,   :scale .== "log"), :wis)

println("\n===== log-scale WIS by model (aggregated over horizons & ", length(wins), " origins) =====")
show(by_m_log, allcols = true); println()
println("\n===== log-scale WIS by model × horizon (aggregated over origins) =====")
show(by_mh_log, allcols = true); println()

# native sample-CRPS cross-check, averaged over origins
crps_df = sort(combine(groupby(DataFrame(crps_rows), :model), :mean_crps => mean => :mean_crps),
               :mean_crps)
println("\nmean native CRPS (avg over origins):")
show(crps_df, allrows = true); println()

CSV.write("../res/8j_scores_by_model.csv", scores.by_model)                        # both scales
CSV.write("../res/8j_scores_by_model_horizon.csv", scores.by_model_h)              # both scales × horizon
CSV.write("../res/8j_scores_by_model_date.csv", scores.by_model_dt)                # both scales × origin
CSV.write("../res/8j_scores_by_model_date_horizon.csv", scores.by_model_dt_h)      # both scales × origin × horizon
by_mh_log

In [ ]:
# Diagnostic-visualisation helpers (read-only; loads cached chains without rebuilding the model)
include("8j_viz_utils.jl")   # chain_path, load_transmission_draws, aggregate_supergroups, pick_origins

# Fixed model order + colour palette shared by the diagnostic figures below (the "four ways").
labels4    = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols = [:steelblue, :darkorange, :seagreen, :purple]

In [ ]:
# log-scale WIS by horizon (one line per model), aggregated over all origins
Hn = length(cfg.horizons)
wis_h_fig = plot(; xlabel = "horizon (weeks)", ylabel = "mean log-scale WIS",
                 title = "8j — log-scale WIS by horizon ($(length(wins)) origins)",
                 size = (760, 420), legend = :topleft, xticks = 1:Hn)
for m in unique(by_mh_log.model)
    sub = sort(@subset(by_mh_log, :model .== m), :horizon)
    plot!(wis_h_fig, sub.horizon, sub.wis; marker = :circle, lw = 2, label = m)
end
savefig(wis_h_fig, "../res/8j_wis_by_horizon.png")

# four-ways bar, grouped by horizon (log scale).
# FIX: the old `bar(models, wis; orientation=:h)` mapped the *model index* (0–4) to bar
# length instead of the WIS value, and overflowed the labels. Encode WIS as bar height
# (grouped by horizon) with `groupedbar`; a `(model × horizon)` matrix, labels via `xticks`.
Mwis = [only(@subset(by_mh_log, :model .== m, :horizon .== h).wis)
        for m in labels4, h in 1:Hn]                       # rows = model, cols = horizon
wis_bar = groupedbar(Mwis; bar_position = :dodge,
                     xticks = (1:length(labels4), labels4), xrotation = 20,
                     label = reshape(["h=$h" for h in 1:Hn], 1, :),
                     ylabel = "mean log-scale WIS (lower = better)", legend = :topleft,
                     title = "8j — log-scale WIS by model × horizon",
                     size = (950, 480), bottom_margin = 14Plots.mm, left_margin = 6Plots.mm)
savefig(wis_bar, "../res/8j_wis_four_ways.png")

# log-scale WIS over the forecast period (one line per model) — the full-period skill
by_dt_log = sort(@subset(scores.by_model_dt, :scale .== "log"), [:model, :forecast_date])
wis_t_fig = plot(; xlabel = "forecast origin", ylabel = "mean log-scale WIS",
                 title = "8j — log-scale WIS over the available period",
                 size = (900, 420), legend = :topleft)
for m in unique(by_dt_log.model)
    sub = @subset(by_dt_log, :model .== m)
    plot!(wis_t_fig, sub.forecast_date, sub.wis; lw = 2, marker = :circle, ms = 2, label = m)
end
savefig(wis_t_fig, "../res/8j_wis_over_time.png")
wis_t_fig

In [ ]:
# Log-scale WIS as a time series, separated BY MODEL (one line per config), faceted BY HORIZON.
# Uses the (model × forecast_date × horizon × scale) aggregation added to `score_wis`.
bdth = @subset(scores.by_model_dt_h, :scale .== "log")
d1_panels = Plots.Plot[]
for (k, h) in enumerate(cfg.horizons)
    p = plot(; title = "horizon $h (wk ahead)", titlefontsize = 8, xlabel = "forecast origin",
             ylabel = "mean log-scale WIS", legend = (k == 1 ? :topleft : false),
             legendfontsize = 6, xrotation = 45)
    for (ci, m) in enumerate(labels4)
        s = sort(@subset(bdth, :model .== m, :horizon .== h), :forecast_date)
        plot!(p, s.forecast_date, s.wis; color = model_cols[ci], lw = 1.5,
              marker = :circle, ms = 2, label = m)
    end
    push!(d1_panels, p)
end
wis_dth_fig = plot(d1_panels...; layout = (2, 2), size = (1150, 780),
                   plot_title = "8j — log-scale WIS over time, by horizon",
                   plot_titlefontsize = 11)
savefig(wis_dth_fig, "../res/8j_wis_logscale_by_horizon_over_time.png")
wis_dth_fig

In [ ]:
# Forecast vs observed, evaluated at several origins. Each panel = one forecast origin:
# ONE observed series (8 weeks of history before the origin ++ the 4 realized target weeks)
# overlaid with the FOUR configs' total-infection forecast fans (median + 90% band).
sel = pick_origins(FORECAST_ORIGINS; n = 9)      # evenly-spaced origins across the period
qs_lo, qs_hi = 0.05, 0.95
H = length(cfg.horizons)
fc_panels = Plots.Plot[]
for (pi, origin) in enumerate(sel)
    win   = wins[findfirst(==(origin), FORECAST_ORIGINS)]
    wd    = load_window_data(win; grid = grid)          # observed history (A × all_weeks)
    truth = load_forecast_truth(win; grid = grid)       # observed target weeks (A × H)
    # one continuous observed line: 8 fit weeks (cols smax+1:end) ++ the H forecast weeks
    x_hist = week_mid.(win.fit_weeks)
    y_hist = vec(sum(wd.I_mean[:, (cfg.smax + 1):end]; dims = 1))
    x_fore = week_mid.(win.forecast_weeks)
    y_fore = [sum(truth[:, h]) for h in 1:H]
    p = plot(; title = string(origin), titlefontsize = 7, xrotation = 45,
             legend = (pi == 1 ? :topleft : false), legendfontsize = 5)
    plot!(p, vcat(x_hist, x_fore), vcat(y_hist, y_fore);
          color = :black, lw = 2, marker = :circle, ms = 2, label = "observed")
    vline!(p, [week_mid(win.origin)]; color = :gray, ls = :dash, lw = 1, label = "")
    for (ci, lbl) in enumerate(labels4)
        haskey(fc_store, (origin, lbl)) || continue      # skipped origin×combo → gap
        tot = dropdims(sum(fc_store[(origin, lbl)]; dims = 1); dims = 1)   # H × draws
        med = [median(tot[h, :]) for h in 1:H]
        lo  = [quantile(tot[h, :], qs_lo) for h in 1:H]
        hi  = [quantile(tot[h, :], qs_hi) for h in 1:H]
        plot!(p, x_fore, med; color = model_cols[ci], lw = 1.6,
              ribbon = (med .- lo, hi .- med), fillalpha = 0.10,
              label = (pi == 1 ? lbl : ""))
    end
    push!(fc_panels, p)
end
fc_fig = plot(fc_panels...; layout = (3, 3), size = (1300, 1000),
              plot_title = "8j — total-infection forecast (four ways) vs observed, by origin (90% band)",
              plot_titlefontsize = 11)
savefig(fc_fig, "../res/8j_forecast_vs_observed_panels.png")
fc_fig

In [ ]:
# Fitted transmission structure over the forecast origins (one horizon), from the cached chains.
# susceptibility & infectivity are shown as RATIOS to the 2-15 reference group (2-15 ≡ 1),
# comparing 16-49 and >50; the separable-GP length-scale ρ is shown per config. Faceted 2×2.
H_VIZ = 1                                  # "one horizon" — the direct 1-week-ahead fit
O  = FORECAST_ORIGINS; nO = length(O)
gnames = ["16-49", ">50"]                  # the two groups compared to 2-15 (≡ 1 by construction)

mkratio() = Dict(l => (med = fill(NaN, nO, 2), lo = fill(NaN, nO, 2), hi = fill(NaN, nO, 2))
                 for l in labels4)
susc_store, inf_store = mkratio(), mkratio()
rho_store = Dict(l => (med = fill(NaN, nO), lo = fill(NaN, nO), hi = fill(NaN, nO)) for l in labels4)

for lbl in labels4, (oi, origin) in enumerate(O)
    d = load_transmission_draws(lbl, origin, H_VIZ)     # nothing if chain missing → leaves NaN gap
    d === nothing && continue
    for (V, dst) in ((d.susc, susc_store), (d.inf, inf_store))
        sg = aggregate_supergroups(V, grid.POP)          # ndraws × 3 (2-15, 16-49, >50)
        r  = sg[:, 2:3] ./ sg[:, 1]                      # ratios vs 2-15
        for g in 1:2
            dst[lbl].med[oi, g] = median(r[:, g])
            dst[lbl].lo[oi, g]  = quantile(r[:, g], 0.05)
            dst[lbl].hi[oi, g]  = quantile(r[:, g], 0.95)
        end
    end
    rho_store[lbl].med[oi] = median(d.rho)
    rho_store[lbl].lo[oi]  = quantile(d.rho, 0.05)
    rho_store[lbl].hi[oi]  = quantile(d.rho, 0.95)
end

function ratio_fig(store, ttl, fname)
    ps = Plots.Plot[]
    for (k, lbl) in enumerate(labels4)
        p = plot(; title = lbl, titlefontsize = 8, xlabel = "forecast origin",
                 ylabel = "ratio to 2-15", legend = (k == 1 ? :topright : false),
                 legendfontsize = 6, xrotation = 45)
        # Reference at 1 as a Date-valued series FIRST → establishes the date x-axis.
        # (A leading `hline!` here initialises a numeric axis and collapses the Dates.)
        plot!(p, [first(O), last(O)], [1.0, 1.0]; color = :gray, ls = :dash, label = "")
        for g in 1:2
            m, lo, hi = store[lbl].med[:, g], store[lbl].lo[:, g], store[lbl].hi[:, g]
            plot!(p, O, m; lw = 1.8, marker = :circle, ms = 2, label = gnames[g],
                  ribbon = (m .- lo, hi .- m), fillalpha = 0.15)
        end
        push!(ps, p)
    end
    f = plot(ps...; layout = (2, 2), size = (1150, 780), plot_title = ttl, plot_titlefontsize = 11)
    savefig(f, fname); f
end

susc_fig = ratio_fig(susc_store, "8j — susceptibility ratio to 2-15 (h=$H_VIZ)",
                     "../res/8j_susc_ratio_over_time.png")
inf_fig  = ratio_fig(inf_store,  "8j — infectivity ratio to 2-15 (h=$H_VIZ)",
                     "../res/8j_infectivity_ratio_over_time.png")

# separable-GP shared length-scale ρ (one line per config; single scalar per fit)
rho_panels = Plots.Plot[]
for lbl in labels4
    m, lo, hi = rho_store[lbl].med, rho_store[lbl].lo, rho_store[lbl].hi
    p = plot(O, m; title = lbl, titlefontsize = 8, xlabel = "forecast origin",
             ylabel = "GP length-scale ρ (age-yrs)", lw = 1.8, marker = :circle, ms = 2,
             ribbon = (m .- lo, hi .- m), fillalpha = 0.15, legend = false,
             xrotation = 45, ylims = (0, 50))
    push!(rho_panels, p)
end
rho_fig = plot(rho_panels...; layout = (2, 2), size = (1150, 780),
               plot_title = "8j — separable-GP length-scale ρ over time (h=$H_VIZ)",
               plot_titlefontsize = 11)
savefig(rho_fig, "../res/8j_lengthscale_rho_over_time.png")
rho_fig

## §4 Notes

- **Available period**: the forecast origin is rolled weekly over the full span the current
  datasets support — bounded below by the first inc2prev week (the 12-week fit window) and
  above by the CoMix contact-data end (the iterate needs contacts to origin+3). Scores are
  written per model, per model×horizon, and per model×origin (`res/8j_scores_by_model*.csv`).
- **Four ways** = the 2×2 grid; the headline metric is **log-scale WIS aggregated by horizon**
  across all origins (`res/8j_scores_by_model_horizon.csv`, `scale=="log"`), with
  over/under-prediction & dispersion components, bias, and 50/90% coverage.
- **Parallel, resumable fitting**: per origin the 16 chains are pre-fit concurrently
  (`prefit_chains!`, bounded to a CPU/RAM-balanced `fit_concurrency()`), then reloaded to
  assemble forecasts; chains are cached per (origin, horizon), so a re-run only fits what's
  missing. The raw CoMix tables are read once and reused across every window.
- **Reciprocity + GP smoothing of the mean** (inst/1e): the contact mean is a *symmetric*
  log-rate over the 28 unordered age pairs, `log μ_{i→j} = r_{min,max} + log(popⱼ)` (so
  `popᵢ·μ_{i→j} = popⱼ·μ_{j→i}` exactly), `r` a **separable-RBF GP** over age midpoints
  (70+ → 74.5; shared ρ; non-centred `f = η·L·z`). The neighbourhood NGM stays
  reciprocity-balanced (size-biased C0 is not reciprocal even when μ is).
- **Group-contact duration weight** (inst/1e): `:cnt_mass=="mass"` contacts (no recorded
  duration) get `cfg.w_dur_group = 2.5/240` (fixed now, estimable later); counts unchanged.
- **Neighbourhood-degree NGM among non-zero** (inst/1c, 1d): `C0 = ⟨k²⟩/⟨k⟩ × g`,
  `g = 1/(1−P₀)` (NegBin, floored) / `g = (1−p⁰)` (hurdle-Weibull). Mean NGM (`C0 = ⟨k⟩`) unchanged.
- **Contact-updated iterate** (inst/1d): infections/antibody frozen at each origin; the contact
  matrix is re-estimated each horizon (window ending origin+h−1); renewal stepped one week at a
  time (mean-plugged lags).
- **WIS on a log scale** (inst/1e): `transform_forecasts(fun=log_shift, offset=1)`; both scales
  written, log is the headline. Run uses **Pathfinder** (`USE_NUTS=false`); set `true` for NUTS.
- Remaining lean simplifications: within-window contacts constant per cell (temporal GP is the
  swap-in seam); reference transmission block; 5-day generation interval. Revisit before
  scientific interpretation.